# temporal_cross_validation

Generated with **Lispyn** — originally created by **Eduardo J. Barrios**.

Source: [Lispyn](https://github.com/edujbarrios/lispyn) (Apache-2.0).

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='temporal_cross_validation', assignments=[['md', 'md']], prefix='')


# Chronological splits and walk-forward evaluation

Never mix future observations into training windows.

Use a 2-sample embargo between training, validation and held-out test.

The test set stays untouched until the final evaluation.

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='temporal_cross_validation', assignments=[['temporal-train-val-test', 'temporal-train-val-test'], ['temporal-folds', 'temporal-folds'], ['cross-validate-temporal', 'cross-validate-temporal'], ['train-until', 'train-until'], ['save-cross-validation', 'save-cross-validation']], prefix='')
import torch
x = torch.arange(45, dtype=torch.float32)
y = x * 0.25 + 1.0
dataset = torch.utils.data.TensorDataset(x.unsqueeze(1), y.unsqueeze(1))
from lispyn.temporal import temporal_three_way_dataset
partitions = temporal_three_way_dataset(dataset, validation_fraction=0.2, test_fraction=0.15, gap=2)
history = partitions[0]
validation = partitions[1]
held_out_test = partitions[2]
print('History / validation / held-out test:', len(history), len(validation), len(held_out_test))

def score_window(fold_train, fold_valid, fold, repeat):
    model = torch.nn.Linear(1, 1)
    from lispyn.workflow import train
    result = train(model, fold_train, patience=1, validation=fold_valid, loss_fn=torch.nn.MSELoss(), epochs=2, lr=0.001, batch_size=8, seed=42 + fold)
    return result.history['val_loss'][-1]
from lispyn.temporal import run_temporal_dataset_cv
report = run_temporal_dataset_cv(history, score_window, n_splits=3, test_size=4, gap=1, min_train_size=8, max_train_size=15, metric='val_loss')
print('Mean walk-forward loss:', report['mean'])
print('Embargo samples per fold:', report['gap'])
from lispyn.cv_workflows import save_cv_report
save_cv_report(report, 'runs/temporal_cv_report.json')
